# Supplementary Methods 6-7: validation, error controls, transfer and pooling

This notebook recomputes every result number printed in Supplementary Methods 6 and 7 (v28d lines 661-777) and the additional-confidence rosters of line 633, from the tables in `source_data/supplementary_methods_6_7/`. These sections have no figure panels; the numbers support Supplementary Figs. 1-7 and Supplementary Tables 1-2, which have their own units. Design parameters typed in the text (seeds, thresholds, length windows, software versions) are the named constants at the top of `stats.py`.

Run with `python paper.py build supplementary_methods_6_7` from the repository root.

In [ ]:
from pathlib import Path
import sys
UNIT = Path.cwd()                 # the notebook runs with its own directory as cwd
REPO = UNIT.parents[1]
sys.path[:0] = [str(REPO), str(UNIT)]
SD = REPO / "source_data" / UNIT.name
OUT = REPO / "build" / UNIT.name   # recomputed tables; gitignored
OUT.mkdir(parents=True, exist_ok=True)
import numpy as np
import pandas as pd
import stats
from stats import show

## Sequence controls and lookup specificity (lines 665-667)

One authentic and ten shuffled workflows (five seeds each for all-position and terminal-preserving shuffles) on the representative healthy-stool acquisition. Unit: workflow; query fractions use distinct queries. No test.

In [ ]:
show(stats.stat_scramble(SD))

## Selected-database size (lines 665, 673-675, 713)

Selected FASTA counts for the 726-acquisition roster (15 CAMPI, 58 healthy stool, 306 PREDICT, 347 CapScan) and reservoir entry counts. Line 675 still prints 710; the v28f Supplementary Fig. 1 plots 726.

In [ ]:
show(stats.stat_database_sizes(SD))

## Selection sensitivity (line 679)

58 healthy-stool acquisitions x 19 settings = 1,102 observations. Unit: acquisition-setting observation.

In [ ]:
show(stats.stat_selection_grid(SD))

## CAMPI community recovery and acquisition depth (lines 683-687)

15 CAMPI acquisitions; pooled reference peptides are the union of the 15 per-acquisition reference sets (5-50 residues). Contaminant entries are counted in the supplied CAMPI reference FASTA.

In [ ]:
show(stats.stat_campi(SD))

## Candidate clustering (line 695)

15 acquisitions x 7 arms (unclustered and six identity thresholds).

In [ ]:
show(stats.stat_clustering(SD))

## CAMPI target-entrapment fusion control (lines 713-715)

Diagnostic (E + E0 + 2B + U)/N at peptide q <= 0.01, acquisition counts summed over 15 acquisitions before division, per control seed. Ranges are across the two seeds, not confidence intervals.

In [ ]:
f = stats.stat_fusion(SD)
print(f['fus_table'].to_string(index=False))
show({k: x for k, x in f.items() if k != 'fus_table'})

## Reservoir occurrence of accepted control sequences (line 721)

Unique I/L-equivalent sequences; presence in the 375,333,654-protein reservoir. No test.

In [ ]:
show(stats.stat_label_validity(SD))

## Donor-selected entrapment (lines 727-729)

58 donor-selected and 5 self-selected targets, three FDRBench seeds. Unit: target acquisition.

In [ ]:
show(stats.stat_donor(SD))

## Restricted-reference simulation (line 733)

12 baseline and 12 stress mixtures. Means and 95% percentile bootstrap intervals over whole mixtures, 20,000 draws, seed 2026091051, one random stream consumed in the original order. Unit: synthetic mixture (acquisition).

In [ ]:
show(stats.stat_simulation(SD))

## Healthy-stool transfer and pooling (lines 739-745)

58 transfer targets (10 inter-individual); shuffle-only acceptance in 12 technical replicates at q <= 0.01, 5-50 residues, counted as peptide-acquisition observations.

In [ ]:
show(stats.stat_transfer_pool(SD))

## Common study search space (line 751)

Union of the 58 selected FASTAs; share of quantified features assigned to a representative absent from the acquisition's own FASTA (pooled over 58 acquisitions). Unions of the PREDICT (306) and CapScan (331 in v28d, 347 in v28f) selected FASTAs are summed from per-FASTA new-sequence counts.

In [ ]:
show(stats.stat_common_space(SD))

## Spectrum-split selection (lines 755-761)

Fixed 710-acquisition roster (CapScan 331). PREDICT participants are pseudonymised. The second call shows the 726-acquisition re-run (CapScan 347), which is not printed in v28d.

In [ ]:
show(stats.stat_spectrum_split(SD))
print('--- 726-acquisition re-run (CapScan 347) ---')
show(stats.stat_spectrum_split(SD, 'split_half_acquisitions_726.tsv'))

## Joint healthy-stool search (lines 765-769)

Per-acquisition confidence on the joint search; matched microbial protein groups, medians across 66 technical-replicate and 105 preparation pairs. Leading-species agreement is read from the producer's summary table.

In [ ]:
show(stats.stat_joint(SD))

## 10-fold spectrum cross-fitting (lines 773-777)

591 matched acquisitions (296 PREDICT, 295 CapScan). Medians with participant cluster bootstrap intervals, 10,000 draws, seed 20261004; the region intervals use a separate stream restarted with the same seed.

In [ ]:
cf = stats.stat_crossfit(SD)
print(cf['cf_boot'].to_string(index=False))
print(cf['cf_boot_regions'].to_string(index=False))
show({k: x for k, x in cf.items() if not k.startswith('cf_boot')})
show(stats.stat_same_spectrum(SD))

## Additional-confidence rosters (line 633)

Matched rosters of the PEP sensitivity and the CAMPI fusion searches.

In [ ]:
show(stats.stat_confidence_rosters(SD))

## Checks and recomputed numbers

This unit has no figure panels, so there are no pixel checks. The cell recomputes every NUMBERS.tsv row and every STATS.tsv row and asserts that they equal the committed files, compares the recomputed values with the Source Data package summary sheets copied into source_data (maximum absolute difference must be below 1e-9), then prints the numbers in NUMBERS.tsv order.

In [ ]:
print("pixel checks: not applicable (no figure panels in this unit)")
v = stats.compute_all(SD)
rows = pd.DataFrame(stats.numbers_rows(v))
committed = pd.read_csv(UNIT / "NUMBERS.tsv", sep="\t", dtype=str, keep_default_na=False)
committed = committed[[c for c in committed.columns if not c.startswith(("v30_", "v33_", "v36_", "v42_"))]]  # reference-text columns are checked by tools/check_numbers.py
assert list(rows.columns) == list(committed.columns)
diff = (rows.astype(str).values != committed.values).any(axis=1)
assert not diff.any(), committed[diff]
srows = pd.DataFrame(stats.stats_rows(v, np.__version__))
scommitted = pd.read_csv(UNIT / "STATS.tsv", sep="\t", dtype=str, keep_default_na=False)
scommitted = scommitted[[c for c in scommitted.columns if not c.startswith(("v30_", "v33_", "v36_", "v42_"))]]
assert (srows.astype(str).values == scommitted.values).all()
rows.to_csv(OUT / "NUMBERS_recomputed.tsv", sep="\t", index=False)
srows.to_csv(OUT / "STATS_recomputed.tsv", sep="\t", index=False)
print(f"{len(rows)} NUMBERS rows and {len(srows)} STATS rows equal the committed files")
cmp = stats.compare_with_package(SD, v)
cmp.to_csv(OUT / "package_comparison.tsv", sep="\t", index=False)
print(cmp.groupby("package_sheet").abs_diff.agg(["count", "max"]).to_string())
assert (cmp.abs_diff < 1e-9).all(), cmp[cmp.abs_diff >= 1e-9]
print(rows.status.str.split(":").str[0].value_counts().to_string())
for r in rows.itertuples():
    print(f"{r.v28d_line}\t{r.printed_value}\t{r.recomputed_value}\t{r.status}")

## Updated v42 analyses
These checks use the current input populations. The preceding cells retain the historical calculations.

In [ ]:
from tools import current_crossfit, v42_values
from figures.SFig02 import current as bags_current
from tables.STab6 import current as null_current
from tools.compare_values import same_numeric_value
import csv, re
current_sets = {
    "tools/current_crossfit.py:numbers": current_crossfit.numbers(),
    "tools/v42_values.py:numbers": v42_values.numbers(),
    "figures/SFig02/current.py:numbers": bags_current.numbers(REPO / "source_data/SFig02"),
    "tables/STab6/current.py:numbers": null_current.numbers(REPO / "source_data/STab6"),
}
checked = 0
current_crossfit_checked = 0
for register in REPO.glob("*/*/NUMBERS.tsv"):
    for row in csv.DictReader(register.open(), delimiter="\t"):
        match = re.fullmatch(r"(.+:numbers)\[(.+)\]", row.get("v42_code", ""))
        if match and match[1] in current_sets and row["v42_status"] == "MATCH":
            value = current_sets[match[1]][match[2]]
            assert same_numeric_value(value, row["v42_recomputed_value"]), (str(register), match[2], value)
            checked += 1
            current_crossfit_checked += match[1] == "tools/current_crossfit.py:numbers"
print(f"{checked} current claims reproduced from their current input tables")
assert current_crossfit_checked == 30, current_crossfit_checked
